# Lab 10 — Retrieval-Augmented Generation (RAG)

RAG combines search with generation: **retrieve** relevant context, then **generate** a grounded answer.

| Item | Detail |
|---|---|
| **Pattern** | Embed → Search → Generate |
| **Functions** | `AI_EMBED`, `VECTOR_COSINE_SIMILARITY`, `AI_COMPLETE` |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Full RAG pipeline, context injection, hallucination reduction |

> **Prerequisite:** Run Labs 06 + 09 first to have `wiki_articles` and `wiki_search` available.


## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COMPLETE` (replaces `SNOWFLAKE.CORTEX.COMPLETE`), `AI_EMBED` (replaces `SNOWFLAKE.CORTEX.EMBED_TEXT_768 / EMBED_TEXT_1024`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Build the Retrieval Index

Embed documents for vector search (if not already done in Lab 09).

In [ ]:
CREATE OR REPLACE TABLE rag_knowledge_base AS
SELECT article_id, title, category, body,
    AI_EMBED('e5-base-v2', body) AS embedding
FROM wiki_articles;

## Step 3 — Retrieve Relevant Context

Find the most relevant documents for a given question.

In [ ]:
SET question = 'How does RAG reduce hallucination in AI applications?';

SELECT title, category,
    ROUND(VECTOR_COSINE_SIMILARITY(
        embedding,
        AI_EMBED('e5-base-v2', $question)
    ), 4) AS similarity,
    LEFT(body, 200) AS context_preview
FROM rag_knowledge_base
ORDER BY similarity DESC
LIMIT 3;

## Step 4 — Generate a Grounded Answer

Inject retrieved context into the LLM prompt. The model answers **only** from provided context.

In [ ]:
WITH context AS (
    SELECT body,
        VECTOR_COSINE_SIMILARITY(
            embedding,
            AI_EMBED('e5-base-v2', 'How does RAG reduce hallucination?')
        ) AS similarity
    FROM rag_knowledge_base
    ORDER BY similarity DESC
    LIMIT 2
)
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [
        {'role': 'system', 'content': 'Answer ONLY from the provided context. If the answer is not in the context, say so.'},
        {'role': 'user', 'content': 'Context:\n' || LISTAGG(body, '\n---\n') || '\n\nQuestion: How does RAG reduce hallucination in AI applications?'}
    ],
    {'temperature': 0.1, 'max_tokens': 500}
):choices[0]:messages::STRING AS rag_answer
FROM context;

## Step 5 — Compare: With vs Without RAG

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [{'role': 'user', 'content': 'How does RAG reduce hallucination in AI applications? Answer in 3 sentences.'}],
    {'temperature': 0.1}
):choices[0]:messages::STRING AS answer_without_rag;

Notice how the RAG answer is grounded in specific details from our knowledge base, while the non-RAG answer relies on the model's general training data.

## Key Takeaways

- **RAG = Retrieve + Generate**: embed documents, find relevant ones, inject as context.
- System prompt should instruct the model to answer **only from context**.
- Use low `temperature` (0.1) for factual QA to reduce hallucination.
- RAG with Cortex Search Service (Lab 09) is simpler — no manual embedding needed.
- Always compare RAG vs. non-RAG to validate the improvement.
